In [ ]:
words = open('names.txt', 'r').read().splitlines()

In [ ]:
words[:10]

In [ ]:
freq = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        freq[bigram] = freq.get(bigram, 0) + 1

In [ ]:
freq

In [ ]:
#Sort by count of characters in descending order
sorted(freq.items(), key=lambda kv:-kv[1])[:3]

In [ ]:
import torch

In [ ]:
#Creating 2D array of size 27*27 (26 characters and 1 for <S> and <E>)
N = torch.zeros((27, 27), dtype=torch.int32)

In [ ]:
#Calculate string to int and int to string mapping (look up table for characters)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [ ]:
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        i1 = stoi[ch1]
        i2 = stoi[ch2]
        N[i1, i2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16,16))
plt.imshow(N, cmap='Blues')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr, ha="center", va="bottom", color='gray')
        plt.text(j, i, N[i, j].item(), ha="center", va="top", color='gray')
plt.axis('off');

In [ ]:
#Probability of character to be present in that position
p = N[0].float()
p = p / p.sum()
p

In [ ]:
#Genrating an index at row 0 based on the probability distribution
g = torch.Generator().manual_seed(2147483647)
ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g)
ix

In [ ]:
#Creates a random number between 0 and 1
g = torch.Generator().manual_seed(2147483647)
p = torch.rand(3, generator=g)
#Normalizing the random generated numbers
p = p/p.sum()
p

In [ ]:
#Generating index from the probability distribution of p above (sampling from p)
ix = torch.multinomial(p, num_samples=20, replacement=True, generator=g).tolist()
ix

In [ ]:
#generating the word based on the sampling
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    out = []
    ix = 0
    while True:
        p = N[ix].float()
        p = p/p.sum()
        # p = torch.ones(27)
        # p = p/p.sum()
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    
    print(''.join(out))

In [ ]:
#Creating a 1D matrix of size 1*27, adding 1 for smoothening the model
P = (N+1).float()
P = P / P.sum(1, keepdim=True)

In [ ]:
g = torch.Generator().manual_seed(2147483647)
for i in range(10):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    
    print(''.join(out))

In [ ]:
# GOAL: maximize likelihood of the data w.r.t. model parameters (statistical modeling)
# equivalent to maximizing the log likelihood (because log is monotonic)
# equivalent to minimizing the negative log likelihood
# equivalent to minimizing the average negative log likelihood

# log(a*b*c) = log(a) + log(b) + log(c)

In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
#for w in ["andrejq"]:
  chs = ['.'] + list(w) + ['.']
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    prob = P[ix1, ix2]
    logprob = torch.log(prob)
    log_likelihood += logprob
    n += 1
    #print(f'{ch1}{ch2}: {prob:.4f} {logprob:.4f}')

print(f'{log_likelihood=}')
nll = -log_likelihood
print(f'{nll=}')
print(f'{nll/n}')

In [ ]:
#Creating Training dataset for the neural network
xs, ys = [], []
for w in words:
    ch = ['.'] + list(w) + ['.']
    for c1, c2 in zip(ch, ch[1:]):
        ix1 = stoi[c1]
        ix2 = stoi[c2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)